In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

breast_cancer = load_breast_cancer()

data = breast_cancer.data
features = breast_cancer.feature_names
target = breast_cancer.target

df = pd.DataFrame(data=data, columns=features)
df['target'] = target

print(df.head())
print(df.info())
print(df['target'].unique())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [2]:
from sklearn.model_selection import train_test_split

features = df.drop(columns=['target'])

x_train, x_test, y_train, y_test = train_test_split(features, df['target'], test_size=0.3, random_state=42)

In [3]:
import os 

n_jobs = round(os.cpu_count() / 2)

In [4]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 5, 40),
        'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'bootstrap':        trial.suggest_categorical('bootstrap', [True, False]),
    }
    model = RandomForestClassifier(**params, random_state=42, n_jobs=n_jobs)
    return cross_val_score(model, x_train, y_train, cv=5, scoring='r2').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best CV R²:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

c:\APPLICAZIONI\INFORMATICA\GitHub\CodeAcademy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 4. Best value: 0.817569: 100%|██████████| 50/50 [01:39<00:00,  1.99s/it]

Best CV R²:  0.8176
Best params: {'n_estimators': 557, 'max_depth': 12, 'max_features': 'log2', 'min_samples_leaf': 1, 'bootstrap': False}


In [5]:
best_model = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=n_jobs)
best_model.fit(x_train, y_train)

print(best_model.score(x_test, y_test))

0.9649122807017544
